# Topic 3 — Model Training, Evaluation & Interpretation
### ITPU · Introduction to Machine Learning · Lesson A (live coding companion)

This notebook is the **coding companion** for the lesson. We keep it simple and visual — the goal is to *see* each concept in code, not to write a lot.

**What we'll code today**
1. **Block 1 — What training involves:** the training loop in ~10 lines (predict → loss → adjust → repeat)
2. **Block 2 — Evaluate on unseen data:** train vs test score, and *see* the gap
3. **Block 3 — Overfitting / underfitting:** watch error split as complexity grows
4. **Block 4 — Interpretability:** read a feature-importance / SHAP-style plot

> Runs in Google Colab or Jupyter. Cells build on each other — run top to bottom.


In [ ]:
# Setup — run me first
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)
print("Ready. NumPy", np.__version__)

---
## Block 1 — What *training* involves

**The idea:** a model is `price = w * area + b`. Training means adjusting the **parameters** `w` and `b`
to make the predictions close to the real answers, measured by a **loss** (average squared error).

The **learning rate** is a **hyperparameter** — *we* choose it; the model does not learn it.


In [ ]:
# Tiny dataset: apartment area (m2) -> price (k EUR)
area  = np.array([25, 40, 55, 70, 85, 100, 115], dtype=float)
price = np.array([35, 50, 68, 82, 98, 112, 130], dtype=float)   # roughly price = 0.9*area + 12

# Scale the feature to ~[-1,1] so the gradients for w and b are balanced.
# (Feature scaling is standard practice and keeps training stable.)
area_mean, area_std = area.mean(), area.std()
area_s = (area - area_mean) / area_std

# --- the model (works on the scaled feature) ---
def predict(a_scaled, w, b):
    return w * a_scaled + b

# --- the loss: mean squared error (how wrong we are) ---
def loss(w, b):
    err = predict(area_s, w, b) - price
    return np.mean(err ** 2)

# start with a BAD random guess
w, b = 0.0, 0.0
lr = 0.1            # learning rate  <-- HYPERPARAMETER (you set this)
print(f"Start: w={w:.3f} b={b:.3f}  loss={loss(w,b):.1f}")

In [ ]:
# --- the training loop: predict -> measure -> adjust -> repeat ---
history = []
for step in range(300):
    # gradients of the loss w.r.t. each parameter
    err = predict(area_s, w, b) - price
    grad_w = np.mean(2 * err * area_s)
    grad_b = np.mean(2 * err)
    # ADJUST the parameters a little, against the gradient
    w -= lr * grad_w
    b -= lr * grad_b
    history.append(loss(w, b))

print(f"Trained: w={w:.2f} b={b:.2f}  loss={loss(w,b):.2f}")
print("b ~ 82 = the average price; w ~ 33 = how much price moves per 1 std of area.")
print("Predictions now sit right on the data.")

In [ ]:
# Watch the loss fall — this curve IS 'training'
plt.figure(figsize=(6,3))
plt.plot(history)
plt.title("Loss going down during training")
plt.xlabel("training step"); plt.ylabel("loss (MSE)")
plt.grid(alpha=.3); plt.show()

> **Try it (live):** change `lr` to `0.5` — training is faster. Change it to `1.5` — the loss **explodes**
> (the nudges overshoot). That's why the learning rate is a hyperparameter you must tune.
>
> **Parameters** (`w`, `b`) are learned *inside* the loop. **Hyperparameters** (`lr`, number of steps) are set *outside* it.


---
## Block 2 — Evaluate on **unseen** data

**The mistake:** judging a model by how well it does on the data it *trained* on.
A model can score almost perfectly on training data and still fail on new data.

**The fix:** split the data — train on one part, test on another it has never seen. The **gap** between
train and test performance is your early warning of overfitting.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_moons
from sklearn.metrics import accuracy_score

# a small, slightly noisy classification dataset
X, y = make_moons(n_samples=300, noise=0.35, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

# a very deep tree -> it can memorise the training data
model = DecisionTreeClassifier(max_depth=None, random_state=0)
model.fit(X_train, y_train)

train_acc = accuracy_score(y_train, model.predict(X_train))
test_acc  = accuracy_score(y_test,  model.predict(X_test))
print(f"Training accuracy: {train_acc:.2%}")
print(f"Test accuracy:     {test_acc:.2%}")
print(f"GAP: {train_acc - test_acc:.2%}  <-- big gap = overfitting")

**Discuss:** the training accuracy looks amazing, but the test accuracy is much lower.
The model memorised the training points instead of learning the general pattern. *This is overfitting.*


---
## Block 3 — Overfitting / underfitting & the bias–variance trade-off

Let's *see* it: as we increase model complexity (tree depth), training error keeps falling,
but test error falls, bottoms out, then rises again. The bottom of the test curve is the **sweet spot**.


In [ ]:
depths = range(1, 21)
train_scores, test_scores = [], []
for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_train, y_train)
    train_scores.append(accuracy_score(y_train, m.predict(X_train)))
    test_scores.append(accuracy_score(y_test,  m.predict(X_test)))

plt.figure(figsize=(7,4))
plt.plot(list(depths), train_scores, 'o-', label="train accuracy")
plt.plot(list(depths), test_scores,  'o-', label="test accuracy")
best = int(np.argmax(test_scores))
plt.axvline(list(depths)[best], ls='--', color='green', alpha=.6)
plt.title("Underfit (left)  ->  sweet spot  ->  Overfit (right)")
plt.xlabel("tree depth = model complexity"); plt.ylabel("accuracy")
plt.legend(); plt.grid(alpha=.3); plt.show()
print(f"Best generalisation around depth = {list(depths)[best]} (highest test accuracy).")

**Read the graph together:**
- **Left (shallow tree):** both scores low → **underfitting** (high bias).
- **Right (deep tree):** train ~100%, test drops → **overfitting** (high variance).
- **Green line:** best depth — lowest generalisation error. *This is the bias–variance trade-off.*

> Pair this with the interactive **Bias–Variance Explorer** (HTML) shown on screen.


---
## Block 4 — Interpretability: *reading* a model

Some models are **interpretable by design** (a shallow tree, linear regression); others are opaque.
**Model-agnostic** methods explain any model. The most common in practice is **SHAP**
(how much each feature pushed a prediction up or down).

Here we *read* feature importance — we don't derive the math. Focus on the story the chart tells.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

# a dataset with named features so importances are meaningful
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer()
Xb, yb = data.data, data.target
rf = RandomForestClassifier(n_estimators=200, random_state=0).fit(Xb, yb)

imp = pd.Series(rf.feature_importances_, index=data.feature_names).sort_values()[-10:]
plt.figure(figsize=(7,4))
imp.plot(kind='barh', color='#38bdf8')
plt.title("Top 10 features driving the prediction (global importance)")
plt.xlabel("importance"); plt.tight_layout(); plt.show()
print("Read it: the longest bars are the features the model relies on most.")

**Optional — real SHAP** (if the library is available). SHAP explains a *single* prediction:
which features pushed it toward the outcome, and by how much.


In [ ]:
# Optional: real SHAP values. Uncomment to run (installs on first use in Colab).
# !pip -q install shap
# import shap
# explainer = shap.TreeExplainer(rf)
# sv = explainer.shap_values(Xb[:100])
# shap.summary_plot(sv, Xb[:100], feature_names=data.feature_names)
print("If SHAP is installed, the summary plot shows each feature's push per prediction.")

---
## Block 5 — Case study: *diagnose this model* (team task)

For each scenario, decide: **underfitting, overfitting, or good fit** — and what you'd do next.


In [ ]:
scenarios = [
    ("A", 0.98, 0.62),   # train, test
    ("B", 0.71, 0.69),
    ("C", 0.93, 0.90),
]
print("Scenario | Train | Test | Your diagnosis?")
for name, tr, te in scenarios:
    print(f"   {name}     | {tr:.0%}  | {te:.0%} |  ______________")

# Discuss in your breakout room, then reveal the hint below.

<details><summary><b>Reveal the diagnoses (after discussion)</b></summary>

- **A — Overfitting.** Huge train–test gap. Fix: simpler model, more data, regularisation, or fewer features.
- **B — Underfitting.** Both scores low and close. Fix: more complex model, better features, train longer.
- **C — Good fit.** Both high and close — it generalises. Ship it (and keep monitoring).
</details>

### Wrap-up
- Training = adjust **parameters** to minimise **loss**; **hyperparameters** are set by you.
- Always judge a model on **unseen** data — mind the **gap**.
- **Underfit** = too simple (high bias); **overfit** = too complex (high variance); aim for the sweet spot.
- Prefer **interpretable** models where you can; otherwise explain them with **SHAP**.
